# Afinamiento en Google Colab (E10)

Corre `training.tune` (con `lab.afinar`) en una máquina de Colab y guarda el estudio en tu Google Drive.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno*: **CPU** para afinar. Elige **GPU (T4)** solo si quieres probar la celda 5b.
2. Corre las celdas en orden con *Mayús + Enter*.
3. Deja esta pestaña abierta. Colab corta las sesiones inactivas y las que pasan de su límite (unas 12 h en la versión gratis).
   No pasa nada: el estudio se copia a Drive cada 10 minutos y la celda 6 lo continúa donde quedó.

**Qué no hace**: no cambia la configuración del experimento. Usa los mismos YAML del repo que la laptop.

## 1. Configuración

In [ ]:
import os

ALGO = "ddpg"                        # ddpg, ppo, td3 o sac (uno por sesión de Colab)
FUENTE = "github"                    # "github": clona la rama de abajo. "zip": usa ProyectoFinal-APR-lab.zip de DRIVE_DIR
RAMA = "mario/e9-tune-callbacks"
DRIVE_DIR = "/content/drive/MyDrive/APR-colab"   # aquí queda el estudio (results/optuna)
WORKERS = os.cpu_count()             # procesos en paralelo; ajústalo con lo que diga la celda 5

REPO = "/content/ProyectoFinal-APR"
print(f"Esta máquina tiene {os.cpu_count()} hilos lógicos.")
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "Sin GPU"

## 2. Conectar Google Drive
Te va a pedir que autorices el acceso a tu Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

## 3. Traer el código
- `github`: la rama tiene que estar subida (`git push`). Si ya existe, solo la actualiza.
- `zip`: en la laptop, doble clic en `lab\empaquetar.bat` y sube `ProyectoFinal-APR-lab.zip` a la carpeta `DRIVE_DIR` de Drive.

In [ ]:
if not os.path.exists(REPO):
    if FUENTE == "github":
        !git clone --depth 1 --branch {RAMA} https://github.com/Ctribsz/ProyectoFinal-APR.git {REPO}
    else:
        !unzip -q "{DRIVE_DIR}/ProyectoFinal-APR-lab.zip" -d /content
elif FUENTE == "github":
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1 2>/dev/null || echo "(código del zip)"

## 4. Instalar (unos 3–5 minutos)
Mismas versiones que la laptop (`lab/requirements-lab.txt`), porque cada corrida las guarda en su `meta.json`.
En un entorno con GPU se deja el torch de Colab (trae CUDA). Los avisos de pip sobre otros paquetes de Colab no importan:
todo se corre con `python` en procesos aparte.

In [ ]:
import shutil
if shutil.which("nvidia-smi") is None:
    !pip install -q torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r lab/requirements-lab.txt
!python -c "from envs.balance_bot_env import BalanceBotEnv, load_config; import stable_baselines3, optuna, torch; env = BalanceBotEnv(level=2, config=load_config(2)); env.reset(seed=0); env.step(env.action_space.sample()); print('OK: entorno, SB3', stable_baselines3.__version__, 'Optuna', optuna.__version__, 'torch', torch.__version__)"

## 5. Medir esta máquina (unos 10 minutos)
Da los pasos/s de DDPG y PPO en un proceso y con `WORKERS` procesos de DDPG a la vez. Compáralo con la laptop
(DDPG ~58 pasos/s y PPO ~410 pasos/s en el nivel 2). Si "en total" no sube con más procesos, baja `WORKERS` en la celda 1.
Para solo probar que todo funciona: agrega `--quick`.

In [ ]:
!python -m lab.benchmark --parallel {WORKERS}

### 5b. (Opcional, solo con GPU) ¿La GPU acelera?
Compara `device=cpu` con `device=cuda` para DDPG. **Solo mide**: el afinamiento sigue usando `device` de
`configs/training/common.yaml`. Cambiarlo afecta a los 4 algoritmos, así que se decide en equipo.

In [ ]:
!python -m lab.colab gpu --algo ddpg

## 6. Afinar
Corre las pruebas de `configs/training/tune.yaml` que falten y copia `results/optuna` a Drive cada 10 minutos y al terminar.
- **Si la sesión se cortó:** corre de nuevo las celdas 1 a 4 y luego esta. Trae el estudio de Drive, marca como FAIL las
  pruebas que quedaron a medias (no cuentan para las 40) y sigue con las que faltan.
- **Para detenerlo:** el botón de detener de la celda. El estudio queda en Drive y se puede continuar.
- El avance aparece cada minuto. Los logs de cada worker están en `results/optuna/logs/`.

In [ ]:
!python -m lab.colab afinar --algo {ALGO} --workers {WORKERS} --drive "{DRIVE_DIR}"

## 7. Traer los resultados a la laptop
En Drive, descarga la carpeta `APR-colab/results/optuna` y copia su contenido en `results/optuna` del repo (el `.db`, la
carpeta del estudio y `logs/`). Con eso se analiza el estudio y se elige la mejor configuración (E10).

Para ver el estado aquí mismo:

In [ ]:
!python -c "from lab.afinar import study_paths, status_line; from training.train import load_yaml; from training.tune import TUNE_CONFIG; n, db = study_paths('{ALGO}', load_yaml(TUNE_CONFIG)); print(n, '->', status_line(n, db))"